## Installing dependenices
RAG pipeline:
document loading
embedding models
vector databases
llm model

In [1]:
print("Installing Dependencies \n")

!pip install -q \
    langchain \
    langchain_community \
    langchain_text_splitters \
    langchain_huggingface \
    langchain_groq \
    langchain-google-genai \
    langchain_openai \
    langchain_core \
    faiss_cpu \
    pypdf \
    sentence-transformers \
    transformers \
    torch \
    huggingface_hub \
    groq \
    langsmith \
    python_dotenv \
    tiktoken
print("\Installation Completed")

Installing Dependencies 



<>:4: SyntaxWarning: invalid escape sequence '\I'
<>:4: SyntaxWarning: invalid escape sequence '\I'
C:\Users\Pravallika\AppData\Local\Temp\ipykernel_24492\2509524310.py:4: SyntaxWarning: invalid escape sequence '\I'
  print("\Installation Completed")


\Installation Completed



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
LLM_Provider = 'groq'
LLM_Model = "openai/gpt-oss-120b"
CORPUS_PATH = "./zyro-dynamics-hr-corpus"

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env")

groq_api_key = os.getenv("GROQ_API_KEY")
if groq_api_key:
    print("GROQ API KEY is loaded from .env")
else:
    print("GROQ API KEY is not found. Add it to .env or set GROQ_API_KEY in the environment.")


GROQ API KEY is loaded from .env


In [11]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
CORPUS_PATH = "./zyro-dynamics-hr-corpus"
loader = PyPDFDirectoryLoader(CORPUS_PATH)
documents = loader.load()
print(f"Loaded {len(documents)} documents")

Loaded 39 documents


In [12]:
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(documents)
print(f"Created{len(chunks)} chunks")

Created107 chunks


Embedding Initialization

In [13]:
#HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("Embedding model loaded")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 887.64it/s]


Embedding model loaded


##VectorDB + Retrival
FAISS VECTOR DB

In [14]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(chunks, embedding_model)
#Retrival
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Retriever created")

Retriever created


##LLM Initialization
LLM model_GROQ



In [19]:
from langchain_groq import ChatGroq

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY is missing. Add it to your .env file or export it in the kernel.")

llm_model = ChatGroq(
    model=LLM_Model,
    api_key=groq_api_key,
    temperature=0.5,
    max_tokens=500,
)
print(f"LLM model {LLM_Model} loaded")


LLM model openai/gpt-oss-120b loaded


##RAG CHAIN

In [20]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langsmith import traceable

RAG_PROMPT = ChatPromptTemplate.from_template(
    """You are a helpful assistant. Use the following pieces of context to answer the question at the end. If you don't know the answer, just say you don't know, don't try to make up an answer.

context: {context}
Question: {question}"""
)


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


@traceable(name="rag_chain")
def rag_chain(question: str):
    docs = retriever.invoke(question)
    context = format_docs(docs)
    chain = RAG_PROMPT | llm_model | StrOutputParser()
    answer = chain.invoke({
        "context": context,
        "question": question,
    })
    return {
        "answer": answer,
        "sources": docs,
    }

## Guardials

(To ensure no context is used beyond the corups)

In [21]:
GUARDIAN_PROMPT = ChatPromptTemplate.from_template("""

You are a scope classifier for an HR assistant. Decide whether the question below is something an HR assistant should answer (company leave policy, reimbursement, code of conduct, or other internal HR topics) or something out of scope (general knowledge, coding help, unrelated small talk, etc).
Respond with exactly one word: IN_SCOPE or OUT_OF_SCOPE. Question: {question}
""")

# Backward-compatible alias for earlier misspelling
GUARDIAL_PROMPT = GUARDIAN_PROMPT

REFUSAL_MESSAGE = (
    "I'm an HR assistant and can only help with questions about company HR "
    "policies (leave, reimbursement, code of conduct, etc.). I don't have "
    "information to answer that question."
)


def ask_bot(question: str):
    guardian_chain = GUARDIAN_PROMPT | llm_model | StrOutputParser()
    verdict = guardian_chain.invoke({"question": question}).strip().upper()
    if 'OUT_OF_SCOPE' in verdict:
        return {"answer": REFUSAL_MESSAGE, "sources": []}
    return rag_chain(question)


print("Guardian and RAG chain created")

Guardian and RAG chain created


In [22]:
test_questions = [
    "How do I apply for parental leave?",
]

for i, q in enumerate(test_questions, 1):
    result = ask_bot(q)
    print(f"Q{i}: {q}")
    print(f"A{i}: {result['answer']}")
    if result.get("sources"):
        print(f"Sources: {[doc.metadata.get('source') for doc in result['sources']]}")
    print("-" * 60)

Q1: How do I apply for parental leave?
A1: **Applying for Parental Leave (Maternity or Paternity) at Zyro Dynamics**

Below is a step‑by‑step guide that covers both maternity and paternity leave, using the policies that are currently in force.

---

## 1. Confirm Your Eligibility  

| Leave Type | Minimum Service Required | Other Conditions |
|------------|--------------------------|------------------|
| **Maternity Leave** | ≥ 80 days of service in the 12 months **before** the expected delivery date | – First two live births: 26 weeks paid (12 weeks for a third child). <br>– Up to 8 weeks can be taken **before** the expected delivery date. |
| **Paternity Leave** | No specific service threshold mentioned (any male employee or same‑sex partner) | – 10 consecutive paid days must be taken **within 90 days** of the birth or legal adoption. <br>– One‑time entitlement per child; cannot be split. |

If you meet the above criteria, you are eligible to apply.

---

## 2. Gather Required Docume

In [24]:
import pandas as pd
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

test_df = pd.read_csv("test.csv")
answers = []

for i, row in test_df.iterrows():
    q = row['question']
    result = ask_bot(q)
    answers.append(result["answer"])
    print(f"[{i+1}/{len(test_df)}] {q}\n -> {result['answer']}\n{'-'*60}")

test_df["answer"] = answers
submission = test_df[["question_id", "answer"]].copy()
submission.to_csv("submission.csv", index=False)
submission.head()

[1/20] How does my Earned Leave accrue every month?
 -> Earned Leave (EL) builds up month‑by‑month according to where you are in your employment tenure:

| Employment stage | Accrual rate | When it becomes usable |
|------------------|--------------|------------------------|
| **Probation period** | **0.5 day per month** | The days you earn while on probation are **not** usable until you are confirmed in your role. |
| **After probation – before completing 1 year of service** | **0.5 day per month** (same as during probation) | The accrued balance can be used once you are confirmed. |
| **After you have completed 1 year of continuous service** (and have worked at least 240 days in that year) | **1.25 days per month** | These days are available for use immediately (subject to manager approval). |
| **Once you have the initial 15 days** (earned after the first year) | **1.25 days per month** continues | The monthly accrual keeps adding to your EL balance throughout your employment. |

**

,question_id,answer
0,Q01,Earned Leave (EL) builds up month‑by‑month acc...
1,Q02,I don’t know.
2,Q03,You are entitled to **26 weeks of paid Materni...
3,Q04,Yes—if you take **more than two consecutive da...
4,Q05,Your salary is processed and credited to your ...
